# Road Accidents France — Machine Learning

This notebook builds a classification model to predict injury severity (`grav`) from accident features.

**Approach:** Temporal validation — trained on 2023 data, tested on 2024 data.
This simulates a real production setting where a model trained on past data is used to predict future events.

**Target variable:**
- 1 — Uninjured
- 2 — Killed
- 3 — Hospitalized
- 4 — Minor injury

**Data:** Output of `cleaning.ipynb`

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

df_train = pd.read_csv('accidents_france_2023_clean_for_ML.csv')
df_test = pd.read_csv('accidents_france_2024_clean_for_ML.csv')

print(f"Train set (2023): {df_train.shape}")
print(f"Test set  (2024): {df_test.shape}")

## 1. Class Distribution

The dataset is heavily imbalanced — fatalities represent only ~2.7% of cases.

In [ ]:
print("Class distribution (train):")
print(df_train['grav'].value_counts(normalize=True).round(3))

## 2. Preprocessing

- Drop identifier columns (no predictive value, potential data leakage)
- Encode remaining categorical columns with LabelEncoder
- Fit encoder on both train and test to handle unseen values

In [ ]:
# Separate target
y_train = df_train['grav']
y_test = df_test['grav']

# Drop target and identifier columns
cols_to_drop = ['grav', 'id_usager', 'id_vehicule', 'Num_Acc', 'num_veh_x', 'num_veh_y', 'com', 'datetime']
X_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns])
X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])

# Encode categorical columns
# Fit on both sets to handle values present in test but not in train
for col in X_train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[col], X_test[col]]).astype(str))
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print(f"Features: {X_train.shape[1]}")
print(X_train.columns.tolist())

## 3. Baseline Model

Before training, we establish a naive baseline: always predict the most frequent class.
Any useful model must clearly outperform this baseline.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("=== Dummy Classifier (baseline) ===")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

## 4. Random Forest Classifier

We use `class_weight='balanced'` to compensate for the severe class imbalance (fatalities: 2.7%).

**Temporal validation:** trained on 2023, evaluated on 2024.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ['Uninjured (1)', 'Killed (2)', 'Hospitalized (3)', 'Minor injury (4)']

px.imshow(cm,
          x=labels, y=labels,
          labels=dict(x='Predicted', y='Actual'),
          title='Confusion Matrix — Random Forest (tested on 2024)',
          height=500)

## 6. Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False).head(20).reset_index()
importances.columns = ['feature', 'importance']

px.bar(importances, x='importance', y='feature', orientation='h',
       title='Top 20 Feature Importances (Gini)', height=600)

**Key observations:**
- No single feature dominates — the highest importance (Age) reaches only ~9.4%
- Geographic features (`lat`, `long`, `dep`) rank high, likely capturing local road characteristics
- `secu1` (safety equipment) and `catv` (vehicle type) are among the most informative

The absence of any strongly discriminating feature is consistent with the low recall on fatalities.

## 7. Conclusion

**Results summary:**
- Random Forest: accuracy **65%**, macro F1 **0.48**
- Dummy baseline: accuracy **41%**, macro F1 **0.15**

The Random Forest significantly outperforms the naive baseline, demonstrating genuine predictive value.

**However**, recall on fatalities (class 2) remains very low despite `class_weight='balanced'`.
This is explained by two factors:
1. **Class imbalance:** fatalities represent only 2.7% of the dataset
2. **Missing features:** the most important predictors of road fatalities — blood alcohol level, actual speed at impact, driver fatigue — are not available in the public BAAC dataset

**Temporal validation (2023 → 2024):** Results are consistent across years, suggesting the model generalizes well and is not overfitting to a specific year's patterns.

**Next steps:** A binary classifier (killed vs. not killed) could be explored to focus modeling capacity on the most critical class.